# Modelado — Predicción del rendimiento del maíz

## Proyecto: AgroPredict Colombia
**MIAD · Despliegue de Soluciones Analíticas · Semana 5**

Este cuaderno desarrolla y evalúa los modelos supervisados para estimar el
rendimiento del maíz (t/ha) a nivel municipal, a partir de la tabla limpia
generada en el preprocesamiento.

---

## Decisiones tomadas antes de modelar

### 1. Escenario de predicción: antes de la siembra

Predecimos el rendimiento de un periodo **futuro, antes de que se siembre**. Esto
responde a la pregunta de negocio real: apoyar la planeación agropecuaria
*anticipada*, cuando la secretaría de agricultura aún puede tomar decisiones.

La consecuencia es qué información tenemos disponible al predecir: **solo el
pasado y el contexto del territorio**. No conocemos nada del periodo que estamos
prediciendo, porque todavía no ha ocurrido.

### 2. Qué variables podemos usar (y cuáles no)

El rendimiento se define como producción dividida por área cosechada. Por eso,
usar producción o área cosechada como predictores sería **fuga de información**. El área
sembrada del periodo actual tampoco se puede usar, porque antes de sembrar no
existe.

Las variables válidas son de dos tipos:

- **Contexto del periodo a predecir:** departamento, sistema productivo
  (tecnificado/tradicional), semestre y año.
- **Historia del municipio:** el rendimiento de periodos anteriores.

### 3. Formas de construir la información histórica

- **Estacional:** el mismo semestre del año anterior (A se compara con A, B con B).
  Respeta la estacionalidad.
- **Secuencial:** el periodo inmediatamente anterior, sea cual sea. Usa información
  más reciente, pero mezcla semestres con condiciones distintas.

### 4. Forma de evaluar el conjunto de datos

Como queremos predecir el futuro, la evaluación respeta la línea de tiempo. Los municipios se repiten cada año y esto prodía hacer que el modelo "recuerde" en lugar de predecir.

- **Entrenamiento:** 2019–2023
- **Validación:** 2024 (para elegir modelo e hiperparámetros)
- **Prueba final:** 2025 (se reserva intocado; solo se mira una vez, al final)

El resultado sobre 2025 es el que reportamos, porque es el único año que el modelo
nunca vio de ninguna forma.

> **Nota sobre los datos:** la UPRA cambió su metodología entre 2021 y 2022. Como
> entrenamos con años previos y evaluamos en posteriores, parte del error puede
> venir de ese cambio en la fuente y no del modelo. Lo tendremos presente al
> interpretar los resultados.

### 5. Modelos a comparar

Seguimos una escalera de complejidad creciente, para entender qué aporta cada nivel:

- **Regresión Ridge:** modelo lineal de referencia, interpretable.
- **Random Forest:** captura relaciones no lineales e interacciones.
- **Gradient Boosting / XGBoost:** busca el mejor desempeño predictivo.

Cada combinación de modelo y tipo de información histórica (estacional o secuencial) se registra como un experimento en MLflow, con sus parámetros, métricas (RMSE, MAE, R²) y artefactos, para poder
compararlos de forma ordenada.

## 0.Importación librerias

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os
import mlflow

In [3]:
cwd = Path.cwd().resolve()
try:
    HERE = next(c for c in [cwd, *cwd.parents]
                if (c / ".git").exists() or (c / ".dvc").exists())
except StopIteration:
    raise FileNotFoundError(
        "No encuentro la raíz del repo (.git/.dvc). "
        "Abre el notebook dentro del proyecto agropredict-colombia."
    )

DATOS_PROCESADOS = HERE / "data" / "processed"
ARCHIVO_LIMPIO = DATOS_PROCESADOS / "maiz_preprocesado.parquet"

In [4]:
load_dotenv(HERE / "config" / ".env")
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT", "agropredict-maiz-rendimiento")

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"MLflow apuntando a: {MLFLOW_URI}")
print(f"Experimento: {MLFLOW_EXPERIMENT}")

MLflow apuntando a: http://localhost:5001
Experimento: agropredict-maiz-rendimiento


## 1.Configuración del experimento

Todos los parámetros del modelado se definen aquí. Estos valores se
registrarán en MLflow para que cada ejecución quede documentada.

In [5]:
# Tipo de información histórica: "estacional" (mismo semestre año anterior)
#                               o "secuencial" (semestre inmediatamente anterior)
TIPO_REZAGO = "estacional"

# Separación temporal de los datos
ANIOS_ENTRENAMIENTO = [2019, 2020, 2021, 2022, 2023]
ANIO_VALIDACION = 2024
ANIO_PRUEBA = 2025

# Semilla para reproducibilidad.
SEMILLA = 42

# Resumen de la configuración activa
print("Configuración del experimento:")
print(f"  Tipo de rezago : {TIPO_REZAGO}")
print(f"  Entrenamiento  : {ANIOS_ENTRENAMIENTO}")
print(f"  Validación     : {ANIO_VALIDACION}")
print(f"  Prueba         : {ANIO_PRUEBA}")
print(f"  Semilla        : {SEMILLA}")

Configuración del experimento:
  Tipo de rezago : estacional
  Entrenamiento  : [2019, 2020, 2021, 2022, 2023]
  Validación     : 2024
  Prueba         : 2025
  Semilla        : 42


## 2.Carga de datos

In [6]:
df = pd.read_parquet(ARCHIVO_LIMPIO)
print(f"Tabla limpia: {df.shape[0]:,} filas y {df.shape[1]} columnas")
print("\nColumnas:", df.columns.tolist())
df.head()

Tabla limpia: 20,611 filas y 15 columnas

Columnas: ['cod_departamento', 'departamento', 'cod_municipio', 'municipio', 'desagregacion', 'sistema_productivo', 'anio', 'semestre_b', 'post_cambio_metodologico', 'area_sembrada', 'area_cosechada', 'produccion', 'rendimiento', 'flag_cosechada_mayor_sembrada', 'flag_rendimiento_atipico']


,cod_departamento,departamento,cod_municipio,municipio,desagregacion,sistema_productivo,anio,semestre_b,post_cambio_metodologico,area_sembrada,area_cosechada,produccion,rendimiento,flag_cosechada_mayor_sembrada,flag_rendimiento_atipico
0,05,Antioquia,05002,Abejorral,Maíz amarillo tradicional,tradicional,2021,0,0,51.0,49.0,98.0,2.0,0,0
1,05,Antioquia,05002,Abejorral,Maíz amarillo tradicional,tradicional,2021,1,0,40.0,35.0,70.0,2.0,0,0
2,05,Antioquia,05002,Abejorral,Maíz amarillo tradicional,tradicional,2022,0,1,40.0,40.0,80.0,2.0,0,0
3,05,Antioquia,05002,Abejorral,Maíz amarillo tradicional,tradicional,2022,1,1,35.0,35.0,70.0,2.0,0,0
4,05,Antioquia,05002,Abejorral,Maíz amarillo tradicional,tradicional,2023,0,1,40.0,35.0,70.0,2.0,0,0


### Validación previa:

- Antes de agrupar, confirmamos cuántas filas caen en cada unidad de análisis
(municipio–sistema–año–semestre). Esperamos hasta 2 por unidad, una por cada color
de maíz (amarillo y blanco) dentro del mismo sistema. Más de 2, o una misma
variante repetida, indicaría un duplicado en la fuente que habría que corregir
antes de sumar.
- ¿Existe la misma variante repetida dentro de una unidad de análisis? 


In [7]:
llave = ["cod_departamento", "departamento", "cod_municipio", "municipio",
         "sistema_productivo", "anio", "semestre_b"]

conteo = df.groupby(llave).size()

print("Registros por grano:")
print(conteo.value_counts().sort_index())
print(f"\nMáximo de registros en un mismo grano: {conteo.max()}")
print(f"Granos con más de 1 registro: {(conteo > 1).sum():,} de {len(conteo):,}")

Registros por grano:
1    9259
2    5676
Name: count, dtype: int64

Máximo de registros en un mismo grano: 2
Granos con más de 1 registro: 5,676 de 14,935


In [8]:
duplicados = df.duplicated(subset=llave + ["desagregacion"], keep=False)
print(f"Filas con desagregación duplicada en la misma unidad de análisis: {duplicados.sum()}")

Filas con desagregación duplicada en la misma unidad de análisis: 0


## 3.Agrupar a la unidad de análisis

La tabla limpia tiene una fila por cada variante de maíz (amarillo, blanco, etc.).
Para el modelo queremos una fila por **municipio, sistema, año y semestre**. Agrupamos
y sumamos las áreas y la producción, y recalculamos el rendimiento.

In [9]:
base = (df.groupby(llave, as_index=False).agg(area_cosechada=("area_cosechada", "sum"),
               produccion=("produccion", "sum")))

# Re-cálculo del rendimiento (producción por área cosechada)
base["rendimiento"] = base["produccion"] / base["area_cosechada"]

# Cada unidad de analisis debe tener un identificador único, 
# que se puede construir a partir del año y el semestre y que cada vez sea mayor al anterior.
base["t"] = base["anio"] * 2 + base["semestre_b"]

print(f"Filas tras agrupar: {len(base):,}")
base.head()

Filas tras agrupar: 14,935


,cod_departamento,departamento,cod_municipio,municipio,sistema_productivo,anio,semestre_b,area_cosechada,produccion,rendimiento,t
0,05,Antioquia,05002,Abejorral,tradicional,2019,0,100.0,200.0,2.0,4038
1,05,Antioquia,05002,Abejorral,tradicional,2019,1,75.0,150.0,2.0,4039
2,05,Antioquia,05002,Abejorral,tradicional,2020,0,100.0,200.0,2.0,4040
3,05,Antioquia,05002,Abejorral,tradicional,2020,1,75.0,150.0,2.0,4041
4,05,Antioquia,05002,Abejorral,tradicional,2021,0,98.0,196.0,2.0,4042


## 4. Cómo funciona el rezago (ejemplo)

Para traer el rendimiento del pasado "movemos" cada valor hacia adelante en el
tiempo y luego lo unimos. Generamos dos versiones según cuántos periodos atrás miremos:

- **Estacional:** el mismo semestre del año anterior → 2 periodos atrás.
- **Secuencial:** el semestre inmediatamente anterior → 1 periodo atrás.

Veámoslo con un municipio que tiene tres periodos, en la versión **estacional**:

**Lo que de verdad pasó (`base`):**

| periodo | t | rendimiento |
|---|---|---|
| 2023A | 4046 | 3.0 |
| 2023B | 4047 | 3.5 |
| 2024A | 4048 | 4.0 |

**La copia con el tiempo adelantado 2 periodos:**

| t movido | rendimiento_rezago |
|---|---|
| 4048 | 3.0 |
| 4049 | 3.5 |
| 4050 | 4.0 |

**Al unir las dos por `t`:**

- **2024A (t=4048)** encuentra el 3.0 que venía de 2023A → su rezago es **3.0**
- **2023A (t=4046)** no encuentra nada en la copia → rezago **vacío** (es la primera
  aparición, no tiene pasado)
- **2023B (t=4047)** tampoco encuentra → rezago **vacío**

La versión **secuencial** funciona igual, pero moviendo 1 periodo en vez de 2 (2024A
tomaría el valor de 2023B).

Generamos ambas versiones y las guardamos como archivos separados, para compararlas
en el modelado.

In [10]:
# Generamos las dos versiones de features (estacional y secuencial) y las guardamos
for tipo in ["estacional", "secuencial"]:
    distancia = 2 if tipo == "estacional" else 1

    # "Movemos" el rendimiento hacia adelante y lo unimos como rezago
    historia = base[["cod_municipio", "sistema_productivo", "t", "rendimiento"]].copy()
    historia["t"] = historia["t"] + distancia
    historia = historia.rename(columns={"rendimiento": "rendimiento_rezago"})

    features = base.merge(historia, on=["cod_municipio", "sistema_productivo", "t"], how="left")

    salida = DATOS_PROCESADOS / f"features_{tipo}.parquet"
    features.to_parquet(salida, index=False)

    con_rezago = features["rendimiento_rezago"].notna().sum()
    print(f"[{tipo}] {len(features):,} filas, {con_rezago:,} con rezago -> {salida.name}")

[estacional] 14,935 filas, 12,158 con rezago -> features_estacional.parquet
[secuencial] 14,935 filas, 12,145 con rezago -> features_secuencial.parquet
